# 01 — Pré-processamento

Este notebook concentra **todo o tratamento dos dados brutos**.
A partir dele são gerados os *DataFrames* intermediários, salvos em `outputs/`,
que serão consumidos pelos demais notebooks de análise.

## Etapas
1. Carregar os dados brutos (`whatsapp_base_disparo_mascarado` e `whatsapp_dim_telefone_mascarado`).
2. Corrigir tipos (IDs/telefones mascarados como `int64` → `uint64` → `str`).
3. Explodir a coluna `telefone_aparicoes` (lista de aparições por telefone) em formato tabular.
4. Mapear `id_sistema` → nomes legíveis (`Sistema A`, `Sistema B`, …).
5. Inspeções de qualidade no dataset de aparições (nulos, datas inválidas).
6. Sanity check de disparos inconsistentes em `df_base_disparo` (aplicado na origem).
7. Construir bases auxiliares para análises:
   - `df_disparos_sistema`: junção `base_disparo` × `dim_telefone` por sistema.
   - `df_disparos_atualidade`: idem, acrescido da `registro_data_atualizacao` mais recente e `dias_desde_atualizacao`.
   - `df_taxa_read`: taxa de leitura por telefone (independente do sistema), consumido pelo *scoring*.
8. Persistir tudo em `outputs/processed/` (formato Parquet/CSV).

## Imports e configurações

In [141]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

DATA_DIR = Path('../data')
OUT_DIR = Path('../outputs/processed')
MAPPING_DIR = Path('../mapping')

OUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregamento dos dados brutos

In [142]:
df_base_disparo = pd.read_parquet(DATA_DIR / 'whatsapp_base_disparo_mascarado')
df_dim_telefone = pd.read_parquet(DATA_DIR / 'whatsapp_dim_telefone_mascarado')

print(f'base_disparo : {df_base_disparo.shape}')
print(f'dim_telefone : {df_dim_telefone.shape}')

base_disparo : (392921, 16)
dim_telefone : (283289, 11)


In [143]:
df_base_disparo.head()

,id_conta,id_hsm,id_disparo,id_sessao,cpf,id_target,contato_telefone,categoria_hsm,ambiente,criacao_envio_datahora,envio_datahora,falha_datahora,descricao_falha,indicador_falha,id_status_disparo,status_disparo
0,-9142586270102516767,-7590735594542127444,-2317524427909960986,NaN,-5.220753e+18,9206664160911476254,2824089259510570290,Utilidade,prod,2025-10-07 13:14:53.037,2025-10-07 13:15:17.204,NaT,NaN,False,1,processing
1,-9142586270102516767,-3785213779161126614,-6855906542267037066,NaN,-7.869927e+18,6874603298726099113,-4599056651977889342,Utilidade,prod,2025-10-25 10:56:24.194,2025-10-25 10:56:45.028,NaT,NaN,False,1,processing
2,-9142586270102516767,-9203817354237048492,-4043820570593848035,NaN,5.192496e+18,-1874111216941729257,-1731551129526467355,Utilidade,prod,2025-10-29 11:04:14.987,2025-10-29 11:04:45.562,NaT,NaN,False,1,processing
3,-9142586270102516767,-3785213779161126614,-3867536685401812059,NaN,-6.716780e+18,2633882497346372274,-5478890758772712776,Utilidade,prod,2025-11-01 08:31:00.489,2025-11-01 08:31:14.661,NaT,NaN,False,1,processing
4,-9142586270102516767,-3785213779161126614,-1014974390614404669,NaN,7.845383e+18,-7448118375290831604,2224533453719573633,Utilidade,prod,2025-11-01 08:30:58.679,2025-11-01 08:31:18.792,NaT,NaN,False,1,processing


In [144]:
df_dim_telefone.head()

,telefone_ddi,telefone_ddd,telefone_numero,telefone_tipo,telefone_nacionalidade,telefone_qualidade,telefone_aparicoes,telefone_aparicoes_quantidade,telefone_proprietarios_quantidade,telefone_sistemas_quantidade,validacao_telefone
0,55,-1181433720517268842,-6862804366069381626,CELULAR,Brasil,VALIDO,"[{'id_sistema': '1257277410380486863', 'cpf': 5073517428359850284, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': 2024-10-30}]",1,1,1,"{'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}"
1,55,-1181433720517268842,3856002700049294556,CELULAR,Brasil,VALIDO,"[{'id_sistema': '3094574413675758272', 'cpf': 4935468162812723950, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': 2023-09-01}]",1,1,1,"{'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}"
2,55,-1181433720517268842,8067166217402075300,CELULAR,Brasil,VALIDO,"[{'id_sistema': '3094574413675758272', 'cpf': -4330058083945834430, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': 2025-10-08}]",1,1,1,"{'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}"
3,55,-1181433720517268842,1478899732613896317,CELULAR,Brasil,VALIDO,"[{'id_sistema': '4458959843028638627', 'cpf': 1403367098613331454, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': 2025-02-22}]",1,1,1,"{'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}"
4,55,-1181433720517268842,-6105815728808818158,CELULAR,Brasil,VALIDO,"[{'id_sistema': '1257277410380486863', 'cpf': 5453937523093464096, 'proprietario_tipo': 'CPF', 'registro_data_atualizacao': 2023-06-06}]",1,1,1,"{'ddd_valido_br': True, 'formato_valido': True, 'padrao_suspeito': False, 'padrao_invalido': False}"


### Inspeções iniciais
- A maioria dos telefones recebe **apenas 1 disparo**.
- Há **5 status de disparo** distintos e **nenhum nulo** em `status_disparo`.
- `telefone_aparicoes` é uma **lista de dicionários** (várias aparições por telefone).

In [145]:
display(df_base_disparo['contato_telefone'].value_counts().describe())
display(df_base_disparo['status_disparo'].value_counts(normalize=True))
display(df_dim_telefone['telefone_aparicoes'].map(len).describe())

count    294903.000000
mean          1.332374
std           0.843108
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         174.000000
Name: count, dtype: float64

status_disparo
read          0.691147
delivered     0.217359
failed        0.069370
sent          0.014082
processing    0.008042
Name: proportion, dtype: float64

count    283289.000000
mean          5.400040
std         210.982556
min           1.000000
25%           2.000000
50%           3.000000
75%           6.000000
max       76480.000000
Name: telefone_aparicoes, dtype: float64

## 2. Correção de tipos (hashes mascarados)

IDs e telefones são **hashes mascarados** armazenados como `int64`.
Valores acima de 2⁶³ transbordam para negativos (ex.: `-4599056651977889342`).
Convertemos `int64 → uint64 → str` para obter o identificador correto e evitar
operações numéricas indevidas.

In [146]:
def fix_hash(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Converte colunas de hash mascarado (int64 com sinal) para string uint64."""
    for col in cols:
        s = pd.to_numeric(df[col], errors='coerce')
        mask_na = s.isna()
        s_valid = s[~mask_na].astype(np.int64).astype(np.uint64).astype(str)
        df[col] = None
        df.loc[~mask_na, col] = s_valid
    return df


COLS_DISPARO = ['id_conta', 'id_hsm', 'id_disparo', 'id_target',
                'contato_telefone', 'cpf', 'id_sessao']
COLS_TELEFONE = ['telefone_ddd', 'telefone_numero']

df_base_disparo = fix_hash(df_base_disparo, COLS_DISPARO)
df_dim_telefone = fix_hash(df_dim_telefone, COLS_TELEFONE)

print('Amostra contato_telefone após correção:')
print(df_base_disparo['contato_telefone'].head())

Amostra contato_telefone após correção:
0     2824089259510570290
1    13847687421731662274
2    16715192944183084261
3    12967853314936838840
4     2224533453719573633
Name: contato_telefone, dtype: object


## 3. Explosão de `telefone_aparicoes`

Cada linha de `dim_telefone` contém uma **lista de aparições** do telefone em
diferentes sistemas. Explodimos em formato tabular (uma linha por aparição).

In [147]:
# Garante listas Python (e não ndarrays) antes do explode
df_dim_telefone['telefone_aparicoes'] = df_dim_telefone['telefone_aparicoes'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

df_aparicoes = df_dim_telefone.explode('telefone_aparicoes').copy()

for campo in ['id_sistema', 'cpf', 'proprietario_tipo', 'registro_data_atualizacao']:
    df_aparicoes[campo] = df_aparicoes['telefone_aparicoes'].apply(lambda x: x.get(campo))

df_aparicoes = (
    df_aparicoes[['id_sistema', 'telefone_ddi', 'telefone_ddd', 'telefone_numero',
                  'cpf', 'proprietario_tipo', 'registro_data_atualizacao']]
    .drop_duplicates()
)

df_aparicoes = fix_hash(df_aparicoes, ['cpf', 'id_sistema'])
print(f'df_aparicoes shape: {df_aparicoes.shape}')
df_aparicoes.head()

df_aparicoes shape: (1529380, 7)


,id_sistema,telefone_ddi,telefone_ddd,telefone_numero,cpf,proprietario_tipo,registro_data_atualizacao
0,1257277410380486863,55,17265310353192282774,11583939707640169990,5073517428359850284,CPF,2024-10-30
1,3094574413675758272,55,17265310353192282774,3856002700049294556,4935468162812723950,CPF,2023-09-01
2,3094574413675758272,55,17265310353192282774,8067166217402075300,14116685989763717186,CPF,2025-10-08
3,4458959843028638627,55,17265310353192282774,1478899732613896317,1403367098613331454,CPF,2025-02-22
4,1257277410380486863,55,17265310353192282774,12340928344900733458,5453937523093464096,CPF,2023-06-06


### Prova da chave primária de `df_aparicoes`

Verificamos qual combinação de colunas identifica unicamente cada aparição.
Conclusão: **`(id_sistema, telefone_numero, cpf)` é chave** — nesse dataset,
usar `telefone_numero` puro ou `telefone_completo` (`ddi+ddd+numero`) não muda
a contagem de duplicatas.

In [148]:
# telefone completo (ddi + ddd + numero) só para comparação
telefone_completo = (
    df_aparicoes['telefone_ddi'].astype(str)
    + df_aparicoes['telefone_ddd'].astype(str)
    + df_aparicoes['telefone_numero'].astype(str)
)

dups_sis_tel = df_aparicoes.duplicated(subset=['id_sistema', 'telefone_numero']).sum()
dups_sis_tel_completo = df_aparicoes.assign(_t=telefone_completo).duplicated(
    subset=['id_sistema', '_t']
).sum()
dups_sis_tel_cpf = df_aparicoes.duplicated(
    subset=['id_sistema', 'telefone_numero', 'cpf']
).sum()

# (sistema, telefone) com múltiplas datas de atualização
qtd_multi_datas = (
    df_aparicoes.groupby(['id_sistema', 'telefone_numero'])['registro_data_atualizacao']
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

print(f'Total de linhas .................: {len(df_aparicoes)}')
print(f'id_sistema únicos ...............: {df_aparicoes["id_sistema"].nunique()}')
print(f'telefone_numero únicos ..........: {df_aparicoes["telefone_numero"].nunique()}')
print(f'Duplicatas (id_sistema + numero) : {dups_sis_tel}')
print(f'Duplicatas (id_sistema + completo): {dups_sis_tel_completo}')
print(f'Duplicatas (id_sistema + numero + cpf): {dups_sis_tel_cpf}  ← chave primária')
print(f'(id_sistema, telefone) com múltiplas datas: {qtd_multi_datas}')

Total de linhas .................: 1529380
id_sistema únicos ...............: 6
telefone_numero únicos ..........: 283289
Duplicatas (id_sistema + numero) : 936643
Duplicatas (id_sistema + completo): 936643
Duplicatas (id_sistema + numero + cpf): 0  ← chave primária
(id_sistema, telefone) com múltiplas datas: 37830


## 4. Mapeamento de sistemas

São apenas **6 sistemas únicos**. Renomeamos para nomes legíveis (`Sistema A` … `Sistema F`)
e persistimos o mapeamento em `mapping/mapping_sistemas.csv`.

In [149]:
ids_sistema = df_aparicoes['id_sistema'].unique()
mapping_sistemas = {id_: f'Sistema {chr(65 + i)}' for i, id_ in enumerate(ids_sistema)}

df_aparicoes['sistema_nome'] = df_aparicoes['id_sistema'].map(mapping_sistemas)

df_mapping = pd.DataFrame(mapping_sistemas.items(), columns=['id_sistema', 'sistema_nome'])
df_mapping.to_csv(MAPPING_DIR / 'mapping_sistemas.csv', index=False)
df_mapping

,id_sistema,sistema_nome
0,1257277410380486863,Sistema A
1,3094574413675758272,Sistema B
2,4458959843028638627,Sistema C
3,15689377901922904472,Sistema D
4,13742676811738960007,Sistema E
5,18313131241423355789,Sistema F


## 5. Inspeções no dataset de aparições
- Em mediana e em média, um telefone aparece em **2 sistemas** distintos.
- A coluna `registro_data_atualizacao` tem **muitos nulos no Sistema F (~91%)** e algum no Sistema A (~24%).
- Há datas inválidas (`1899-08-07`) no Sistema F que precisarão ser filtradas nas análises temporais.

In [150]:
display(df_aparicoes.groupby('telefone_numero')['sistema_nome'].nunique().describe())

nulos = df_aparicoes['registro_data_atualizacao'].isna().groupby(df_aparicoes['sistema_nome']).sum()
total = df_aparicoes.groupby('sistema_nome').size()
pct_nulos = (nulos / total * 100).rename('pct_data_nula')
display(pct_nulos)

display(df_aparicoes['registro_data_atualizacao'].value_counts().sort_index().head(10))


count    283289.000000
mean          2.092340
std           0.941366
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max           6.000000
Name: sistema_nome, dtype: float64

sistema_nome
Sistema A    23.745899
Sistema B     0.000000
Sistema C     0.000000
Sistema D     0.000000
Sistema E     0.000000
Sistema F    90.711136
Name: pct_data_nula, dtype: float64

registro_data_atualizacao
1899-08-07    213
1993-05-01      2
1994-05-02      1
1995-10-21      1
1998-10-14      1
1998-10-19      1
1999-04-23      1
1999-05-03      1
1999-05-20      1
1999-07-01      1
Name: count, dtype: int64

In [151]:
# garantir tipo datetime (se ainda não estiver)
df_aparicoes['registro_data_atualizacao'] = pd.to_datetime(
    df_aparicoes['registro_data_atualizacao'], errors='coerce'
)

data_invalida = pd.Timestamp('1899-08-07')

# contagem por sistema
count_1899 = (
    (df_aparicoes['registro_data_atualizacao'] == data_invalida)
    .groupby(df_aparicoes['sistema_nome'])
    .sum()
)

# percentual por sistema
pct_1899 = (count_1899 / total * 100).rename('pct_data_1899')

display(count_1899.rename('qtd_data_1899'))
display(pct_1899)

sistema_nome
Sistema A      0
Sistema B      0
Sistema C      0
Sistema D      0
Sistema E      0
Sistema F    213
Name: qtd_data_1899, dtype: int64

sistema_nome
Sistema A    0.000000
Sistema B    0.000000
Sistema C    0.000000
Sistema D    0.000000
Sistema E    0.000000
Sistema F    0.035667
Name: pct_data_1899, dtype: float64

## 6. Sanity check de disparos inconsistentes

Removemos da **origem** (`df_base_disparo`) registros que possuem `falha_datahora`
preenchida e ao mesmo tempo `status_disparo` em `delivered`/`read`
(estados terminais sem falha). Aplicar o filtro aqui garante que **todos os
datasets derivados** (`df_disparos_sistema` e `df_disparos_atualidade`) herdem
a base já limpa, sem repetir a regra em cada *merge*.

In [152]:
mask_inconsistente = (
    df_base_disparo['falha_datahora'].notna()
    & df_base_disparo['status_disparo'].isin(['delivered', 'read'])
)
print(f'Removendo {mask_inconsistente.sum()} disparos inconsistentes de df_base_disparo')
df_base_disparo = df_base_disparo[~mask_inconsistente].copy()
print(f'df_base_disparo shape: {df_base_disparo.shape}')

Removendo 4 disparos inconsistentes de df_base_disparo
df_base_disparo shape: (392917, 16)


## 7. Bases auxiliares para análise

### 7.1. `df_disparos_sistema` — análise de qualidade por sistema e qualidade por ddd
*Inner join* entre `df_base_disparo` (já saneada na seção 6) e os pares únicos
`(sistema, telefone)` de `df_aparicoes`.

In [ ]:
df_pares_sistema = (
    df_aparicoes[['sistema_nome', 'telefone_numero', 'telefone_ddd', 'id_sistema']]
    .dropna()
    .drop_duplicates()
)

df_disparos_sistema = df_base_disparo.merge(
    df_pares_sistema,
    left_on='contato_telefone',
    right_on='telefone_numero',
    how='inner',
)

print(f'df_disparos_sistema shape: {df_disparos_sistema.shape}')
cob = df_disparos_sistema['contato_telefone'].nunique() / df_base_disparo['contato_telefone'].nunique()
print(f'Cobertura de telefones: {cob:.2%}')

df_disparos_sistema shape: (748675, 19)
Cobertura de telefones: 85.62%


### 7.2. `df_disparos_atualidade` — análise de janela de atualidade
Para cada par `(sistema, telefone)`, mantemos a **data de atualização mais recente**
(filtrando datas inválidas anteriores a 1900) e calculamos `dias_desde_atualizacao`.
Em seguida juntamos com a base de disparos.

In [154]:
df_aparicoes['registro_data_atualizacao'] = pd.to_datetime(
    df_aparicoes['registro_data_atualizacao'], errors='coerce'
)

df_atualizacao_latest = (
    df_aparicoes
    .dropna(subset=['registro_data_atualizacao'])
    .sort_values('registro_data_atualizacao')
    .groupby(['sistema_nome', 'telefone_numero'], as_index=False)
    .tail(1)
)

# remove datas claramente inválidas (anos < 1900)
df_atualizacao_latest = df_atualizacao_latest[
    df_atualizacao_latest['registro_data_atualizacao'].dt.year > 1900
].copy()

hoje = pd.Timestamp.today().normalize()
df_atualizacao_latest['dias_desde_atualizacao'] = (
    hoje - df_atualizacao_latest['registro_data_atualizacao']
).dt.days

print(f'df_atualizacao_latest shape: {df_atualizacao_latest.shape}')

df_atualizacao_latest shape: (498078, 9)


In [155]:
df_disparos_atualidade = df_base_disparo.merge(
    df_atualizacao_latest[['id_sistema', 'sistema_nome', 'telefone_numero',
                           'registro_data_atualizacao', 'dias_desde_atualizacao']],
    left_on='contato_telefone',
    right_on='telefone_numero',
    how='inner',
)

# remove disparos anteriores à data de atualização registrada (inconsistência temporal)
mask_temporal = (
    df_disparos_atualidade['envio_datahora']
    >= df_disparos_atualidade['registro_data_atualizacao']
)
print(f'Removendo {(~mask_temporal).sum()} disparos anteriores à atualização')
df_disparos_atualidade = df_disparos_atualidade[mask_temporal].copy()

print(f'df_disparos_atualidade shape: {df_disparos_atualidade.shape}')

Removendo 1388 disparos anteriores à atualização
df_disparos_atualidade shape: (637671, 21)


### 7.3. `df_taxa_read` — taxa de leitura por telefone
Auxiliar consumido pelo algoritmo de *scoring* (`PhoneScorer`).
Agrega **todos os disparos por telefone** (independente do sistema) e calcula a
taxa de leitura (`read / total`).

- **Granularidade**: 1 linha por `telefone`.
- **Campos**: `telefone`, `total_envios`, `total_reads`, `taxa_read` (∈ [0, 1]).

In [156]:
df_taxa_read = (
    df_base_disparo
    .assign(is_read=(df_base_disparo['status_disparo'] == 'read').astype(int))
    .groupby('contato_telefone', as_index=False)
    .agg(total_envios=('status_disparo', 'size'),
         total_reads=('is_read', 'sum'))
    .rename(columns={'contato_telefone': 'telefone'})
)

df_taxa_read['taxa_read'] = df_taxa_read['total_reads'] / df_taxa_read['total_envios']

print(f'df_taxa_read shape: {df_taxa_read.shape}')
df_taxa_read.head()

df_taxa_read shape: (294901, 4)


,telefone,total_envios,total_reads,taxa_read
0,10000084441235377846,1,1,1.0
1,10000110552567836288,1,0,0.0
2,10000135852850332451,2,2,1.0
3,10000156218966059013,1,1,1.0
4,10000201787338295860,1,1,1.0


### 7.4. `df_cpf_telefones` — entrada principal do `PhoneScorer`

DataFrame **enxuto** consumido pelo algoritmo de *scoring*. Para cada CPF,
lista os telefones associados e, por linha, identifica **o sistema de origem**
e **a data de atualização** daquele telefone.

- **Granularidade**: 1 linha por `(cpf, telefone, id_sistema, data_atualizacao)`.
- **Campos**: `cpf`, `telefone`, `id_sistema`, `data_atualizacao`.

In [157]:
df_cpf_telefones = (
    df_aparicoes
    .loc[:, ['cpf', 'telefone_numero', 'id_sistema', 'registro_data_atualizacao']]
    .rename(columns={
        'telefone_numero': 'telefone',
        'registro_data_atualizacao': 'data_atualizacao',
    })
    .dropna(subset=['cpf', 'telefone', 'id_sistema'])
    .reset_index(drop=True)
)

# datetime garantido
df_cpf_telefones['data_atualizacao'] = pd.to_datetime(
    df_cpf_telefones['data_atualizacao'], errors='coerce'
)

mask_data_ok = (
    df_cpf_telefones['data_atualizacao'].isna()
    | (df_cpf_telefones['data_atualizacao'].dt.year > 1900)
)
df_cpf_telefones = df_cpf_telefones[mask_data_ok].reset_index(drop=True)

# diagnósticos
print(f'df_cpf_telefones shape: {df_cpf_telefones.shape}')
print(f'CPFs únicos    : {df_cpf_telefones["cpf"].nunique()}')
print(f'Telefones únicos: {df_cpf_telefones["telefone"].nunique()}')
print(f'Linhas com data nula: {df_cpf_telefones["data_atualizacao"].isna().sum()}')

df_cpf_telefones.head()

df_cpf_telefones shape: (1529167, 4)
CPFs únicos    : 1221360
Telefones únicos: 283286
Linhas com data nula: 617933


,cpf,telefone,id_sistema,data_atualizacao
0,5073517428359850284,11583939707640169990,1257277410380486863,2024-10-30
1,4935468162812723950,3856002700049294556,3094574413675758272,2023-09-01
2,14116685989763717186,8067166217402075300,3094574413675758272,2025-10-08
3,1403367098613331454,1478899732613896317,4458959843028638627,2025-02-22
4,5453937523093464096,12340928344900733458,1257277410380486863,2023-06-06


## 8. Persistência dos *DataFrames* processados
Os arquivos salvos em `outputs/processed/` e seu propósito:

1. **`base_disparo.parquet`** — base de disparos original já com os IDs/telefones corrigidos (hashes `int64 → uint64 → str`) e **sanidade aplicada** (removidos disparos com `falha_datahora` preenchida e status `delivered`/`read`). Uma linha por disparo.

2. **`aparicoes_telefone.parquet`** — `telefone_aparicoes` explodido: uma linha por aparição `(id_sistema, telefone, cpf, data_atualizacao)`, com `sistema_nome` legível. Chave: `(id_sistema, telefone_numero, cpf)`.

3. **`atualizacao_latest.parquet`** — para cada par `(sistema_nome, telefone_numero)`, a `registro_data_atualizacao` mais recente (datas < 1900 removidas) e `dias_desde_atualizacao` em relação a hoje.

4. **`disparos_sistema.parquet`** — *inner join* de `base_disparo` (já saneada) × pares únicos `(sistema, telefone)`. Base para as análises em 02 e 05.

5. **`disparos_atualidade.parquet`** — `base_disparo` (já saneada) enriquecida com `id_sistema`, `registro_data_atualizacao` e `dias_desde_atualizacao`, filtrando disparos anteriores à última atualização. Base para análise 03 da janela de atualidade.

6. **`taxa_read_telefone.csv`** — agregação por telefone (independente do sistema) com `total_envios`, `total_reads` e `taxa_read` (∈ [0, 1]). Consumido pelo algoritmo de *scoring* (`PhoneScorer`).

7. **`cpf_telefones.parquet`** — entrada principal do `PhoneScorer`. Uma linha por `(cpf, telefone, id_sistema)`, com `data_atualizacao`. Repetições preservadas de propósito (mesmo telefone em vários sistemas/datas) — o algoritmo consolida internamente. Inclui também `total_envios` e `total_reads` para análise da taxa de envio por cpf atual.

In [159]:
# DataFrames base
df_base_disparo.to_parquet(OUT_DIR / 'base_disparo.parquet', index=False)
df_aparicoes.to_parquet(OUT_DIR / 'aparicoes_telefone.parquet', index=False)
df_atualizacao_latest.to_parquet(OUT_DIR / 'atualizacao_latest.parquet', index=False)

# Bases já preparadas para cada análise
df_disparos_sistema.to_parquet(OUT_DIR / 'disparos_sistema.parquet', index=False)
df_disparos_atualidade.to_parquet(OUT_DIR / 'disparos_atualidade.parquet', index=False)
df_taxa_read.to_csv(OUT_DIR / 'taxa_read_telefone.csv', index=False)

# Entrada principal do PhoneScorer (com taxa de leitura por CPF)
df_cpf_telefones.to_parquet(OUT_DIR / 'cpf_telefones.parquet', index=False)

print('Arquivos salvos em', OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob('*')):
    print(f'  - {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')

Arquivos salvos em C:\Users\mwuillau\OneDrive - Nokia\IPLAN\desafio-cientista-dados-pleno-campanhas\outputs\processed
  - aparicoes_telefone.parquet  (32.6 MB)
  - atualizacao_latest.parquet  (17.3 MB)
  - base_disparo.parquet  (33.2 MB)
  - cpf_telefones.parquet  (31.8 MB)
  - disparos_atualidade.parquet  (43.2 MB)
  - disparos_sistema.parquet  (43.9 MB)
  - taxa_read_telefone.csv  (8.7 MB)
